In [2]:
!pip install sentence-transformers


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

C:\Users\siddh\TruthLens\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
with open("corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus size:", len(corpus))

Corpus size: 20812


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|█| 103/103 [00:00<
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vec = model.encode(["test sentence"])
print(vec.shape)

(1, 384)


In [9]:
corpus_embeddings = model.encode(corpus, show_progress_bar=True)

Batches: 100%|█| 651/651 [01:46<00:00,  


In [10]:
dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(corpus_embeddings))

print("FAISS index built")

FAISS index built


In [12]:
faiss.write_index(index, "faiss_index.index")

print("Index saved")

Index saved


In [14]:
index = faiss.read_index("faiss_index.index")

print("Index loaded")

Index loaded


In [15]:
def retrieve_evidence(claim, top_k=5):

    claim_embedding = model.encode([claim])

    distances, indices = index.search(np.array(claim_embedding), top_k)

    results = []

    for i in indices[0]:
        results.append(corpus[i])

    return results

In [16]:
test_claims = [
    "Google was founded in 1998",
    "Shah Rukh Khan is an actor",
    "India conducts population census"
]

for claim in test_claims:
    print("\n🔹 Claim:", claim)

    evidence = retrieve_evidence(claim)

    for e in evidence:
        print(" →", e)


🔹 Claim: Google was founded in 1998
 → Netscape and Amazon (company) were founded in 1994, and Yahoo! in 1995.
 → Google first published exact revenue numbers for YouTube in February 2020 as part of Alphabet's 2019 financial report.
 → On November 13, 2006, YouTube was purchased by Google for 1.65 billion USD (equivalent to $2.44 billion in 2025).
 → Headquartered in San Bruno, California, it is the second-most-visited website in the world, after Google itself.
 → Google introduced an AI Mode at its Google I/O event on 20 May 2025.

🔹 Claim: Shah Rukh Khan is an actor
 → Shah Rukh Khan (pronounced [ˈʃaːɦɾʊx xäːn] ; born as Shahrukh Khan on 2 November 1965), popularly known by the initials SRK, is an Indian actor and film producer renowned for his work in Hindi cinema.
 → == External links == Shah Rukh Khan on X Shah Rukh Khan at IMDb Shah Rukh Khan at Rotten Tomatoes Shah Rukh Khan at Bollywood Hungama
 → Khalid Mohammed from Hindustan Times wrote, "the enterprise belongs to Shah Rukh